In [1]:
#本部分用于测试那些运行起来非常快的指标做快速测试
import cfe
import scanpy as sc

cfe.settings.backend = "python_function"
cfe.logger.setLevel("DEBUG")
import pandas as pd
import numpy as np
from cfe.data import FateAnnData
from cfe.metric.metric_correlation_v2 import calculate_correlation
from cfe.metric.metric_position_predict_v3 import calculate_position_predict
from cfe.metric.metric_featureimp_v3 import (
    calculate_overall_feature_importance,
    calculate_featureimp_cor,
    calculate_featureimp_enrichment,
    fi_ranger_rf_tiny
)
from cfe.metric.cluster_metric_v2 import calculate_mapping_branches, calculate_mapping_milestones,calculate_mapping
from cfe.metric._topology_metric.metric_him import calculate_him
from cfe.metric.topology_metric import calc_isomorphic, calculate_edge_flip

[2025年04月26日 22时18分56秒] INFO                                                                                 
                                          _____     _ _ ______    _       ______            _                      
                                         / ____|   | | |  ____|  | |     |  ____|          | |                     
                                        | |     ___| | | |__ __ _| |_ ___| |__  __  ___ __ | | ___  _ __ ___ _ __  
                                        | |    / _ \ | |  __/ _` | __/ _ \  __| \ \/ / '_ \| |/ _ \| '__/ _ \ '__| 
                                        | |___|  __/ | | | | (_| | ||  __/ |____ >  <| |_) | | (_) | | |  __/ |    
                                         \_____\___|_|_|_|  \__,_|\__\___|______/_/\_\ .__/|_|\___/|_|  \___|_|    
                                                                                     | |                           
                                                                              

In [2]:
# 读取scvelo的pancreas数据
adata = sc.read_h5ad("/home/huang/PyCode/scRNA/data/Pancreas/endocrinogenesis_day15.h5ad")
fadata = cfe.data.FateAnnData.from_anndata(adata)
fadata.layers["expression"] = fadata.layers["spliced"]
fadata.layers["count"] = fadata.layers["spliced"]

basis = "umap"

sc.pp.subsample(fadata, n_obs=500)

fadata

[2025年04月26日 22时18分57秒] DEBUG    Create a FateAnnData object from an existing AnnData object.                
                        DEBUG    Create a FateAnnData object from an existing AnnData object.                      


AnnData object with n_obs × n_vars = 500 × 27998
    obs: 'clusters_coarse', 'clusters', 'S_score', 'G2M_score'
    var: 'highly_variable_genes'
    uns: 'clusters_coarse_colors', 'clusters_colors', 'day_colors', 'neighbors', 'pca', 'cfe'
    obsm: 'X_pca', 'X_umap'
    layers: 'spliced', 'unspliced', 'expression', 'count'

In [3]:
# 手动添加参考里程碑
milestone_network = pd.DataFrame(
    data=[
        ["Ductal", "Ngn3 low EP"],
        ["Ngn3 low EP", "Ngn3 high EP"],
        ["Ngn3 high EP", "Pre-endocrine"],
        ["Pre-endocrine", "Alpha"],
        ["Pre-endocrine", "Beta"],
        ["Pre-endocrine", "Delta"],
        ["Pre-endocrine", "Epsilon"],
        ],
    columns=["from", "to"]
)

fadata.add_trajectory_mannually(milestone_network)

                        DEBUG    FateAnnData add_trajectory                                                        


In [4]:
cluster_key = "milestone_color"
fadata.group_onto_nearest_milestones(cluster_key=cluster_key) # new cluster color

In [5]:
prior_information = {
    "start_id": fadata.obs.index[0],
    "groups_id": fadata.obs[cluster_key].tolist()
}
parameters = {"filter_features": False, "connectivity_cutoff": 0.3}
fadata.add_prior_information(**prior_information)  # add prior information to fadata


In [6]:
method_name_list = ["comp1", "state_comp","paga","cluster_mst","projection_mst"]
for method_name in method_name_list:
        method = cfe.method.FateMethod(method_name=method_name)
        method.infer_trajectory(fadata)

                        DEBUG    FunctionBackend __init__                                                          
                        INFO     Loaded function: <function cf_comp1 at 0x78b040bf3490> from                       
                                 /home/mhn/Code/CellFateExplorer/cfe/method/function/cf_comp1.py                   
                        INFO     method_backend: <cfe.method.fate_function_backend.FunctionBackend object at       
                                 0x78b040335e10>                                                                   
                        DEBUG    FateAnnData add_trajectory                                                        
                        DEBUG    FunctionBackend __init__                                                          
                        INFO     Loaded function: <function cf_state_comp at 0x78b040bf3880> from                  
                                 /home/mhn/Code/CellFateExplorer/cfe/met

In [7]:
parsed_model_name_list = fadata.get_all_model_name() # 解析后的模型名称
model_name_list = fadata.get_all_model_name(parse=False)
parsed_model_name_list, model_name_list

                        WARNING  'ref' is not a valid random_time_string, don't need parse                         


(['ref',
  'comp1-python_function',
  'state_comp-python_function',
  'paga-python_function',
  'cluster_mst-python_function',
  'projection_mst-python_function'],
 ['ref',
  '20250426_221857__comp1-python_function__JeUlnOCx2Q',
  '20250426_221857__state_comp-python_function__FzvtuBBQP4',
  '20250426_221858__paga-python_function__vXS1aE6b7Z',
  '20250426_221900__cluster_mst-python_function__QteUeXEuZa',
  '20250426_221901__projection_mst-python_function__eyjIJd2t6r'])

In [8]:
new_fadata = cfe.data.FateAnnData(
        X=fadata.X,
        obs=fadata.obs,
        uns=fadata.uns
    )
for i in model_name_list: 
    new_fadata.model_name = i
    new_fadata.milestone_wrapper = fadata.trajectory_history_dict[i]['milestone_wrapper']
    new_fadata.add_waypoints()
    # 使用单引号避免冲突
    print(f"{i}'s milestone_network is:\n {new_fadata.milestone_wrapper['milestone_network']}")
    print(f"{i}'s progressions is:\n {new_fadata.milestone_wrapper['progressions']}")

    

prior_information = {
    "start_id": fadata.obs.index[0],
    "groups_id": fadata.obs[cluster_key].tolist()
}
parameters = {"filter_features": False, "connectivity_cutoff": 0.3}
new_fadata.add_prior_information(**prior_information)  # add prior information to fadata

                        DEBUG    FateAnnData add_waypoints                                                         
ref's milestone_network is:
             from             to    length  directed
0         Ductal    Ngn3 low EP  1.671231      True
1    Ngn3 low EP   Ngn3 high EP  8.994165      True
2   Ngn3 high EP  Pre-endocrine  6.586894      True
3  Pre-endocrine          Alpha  6.210582      True
4  Pre-endocrine           Beta  6.141845      True
5  Pre-endocrine          Delta  3.911559      True
6  Pre-endocrine        Epsilon  4.122478      True
ref's progressions is:
              cell_id           from           to  percentage
0   AGCGGTCGTGTATGGG         Ductal  Ngn3 low EP    0.199266
1   TTCTCCTCACAGACTT         Ductal  Ngn3 low EP    0.000000
2   CAGCAGCAGACACGAC         Ductal  Ngn3 low EP    1.000000
3   CGTGTAATCTTTAGTC         Ductal  Ngn3 low EP    0.000000
4   CTGCTGTGTCTTTCAT         Ductal  Ngn3 low EP    0.139954
..               ...            ...          ... 

In [9]:
metrics = cfe.metric.metrics
metrics

,metric_id,plotmath,latex,html,long_name,category,type,perfect,worst,symmetric
0,correlation,cor[dist],\mathit{cor}_{\textrm{dist}},cor<sub>dist</sub>,Geodesic distance correlation,cell positions,specific,1,0.0,True
1,rf_mse,MSE[rf],\mathit{MSE}_{\textit{rf}},MSE<sub>rf</sub>,Random Forest MSE,neighbourhood,specific,0,0.3,False
2,rf_nmse,NMSE[rf],\mathit{NMSE}_{\textit{rf}},NMSE<sub>rf</sub>,Random Forest Normalised MSE,neighbourhood,specific,1,0.0,False
3,rf_rsq,R[rf]^2,R^{2}_{rf},R<sup>2</sup><sub>rf</sub>,Random Forest R²,neighbourhood,specific,1,0.0,False
4,lm_nmse,NMSE[lm],\mathit{NMSE}_{\textit{lm}},NMSE<sub>lm</sub>,Linear regression Normalised MSE,neighbourhood,specific,1,0.0,False
5,lm_mse,MSE[lm],\mathit{MSE}_{\textit{lm}},MSE<sub>lm</sub>,Linear regression MSE,neighbourhood,specific,0,0.3,False
6,lm_rsq,R[lm]^2,R^{2}_{lm},R<sup>2</sup><sub>lm</sub>,Linear regression R²,neighbourhood,specific,1,0.0,False
7,edge_flip,edgeflip,\textrm{edgeflip},edgeflip,Edge flip,topology,specific,1,0.0,True
8,him,HIM,\textrm{HIM},HIM,Hamming-Ipsen-Mikhailov similarity,topology,specific,1,0.0,True
9,isomorphic,isomorphic,\textrm{isomorphic},Isomorphic,isomorphic,topology,specific,1,0.0,True


试试随机森林那些比较慢的指标表现如何

In [10]:
for i in model_name_list:
    print(calculate_featureimp_cor(
            new_fadata,
            "ref",
            i,
            new_fadata.X,
            fi_ranger_rf_tiny()
    ))
    print(calculate_featureimp_enrichment(
            new_fadata,
            "ref",
            i,
            new_fadata.X,
            fi_ranger_rf_tiny()
    ))

{'featureimp_cor': 1.0, 'featureimp_wcor': 1.0}
{'featureimp_ks': 1.0, 'featureimp_wilcox': 1.0}
{'featureimp_cor': 0.6855767779480869, 'featureimp_wcor': 0.7340473938525278}
{'featureimp_ks': 0.0, 'featureimp_wilcox': 0.0}
{'featureimp_cor': 0.7131458304223923, 'featureimp_wcor': 0.7472826417331392}
{'featureimp_ks': 0.0, 'featureimp_wilcox': 0.0}
{'featureimp_cor': 0.8626379316058566, 'featureimp_wcor': 0.9065197807181213}
{'featureimp_ks': 0.0, 'featureimp_wilcox': 0.0}
{'featureimp_cor': 0.9337801122232439, 'featureimp_wcor': 0.9501120453861222}
{'featureimp_ks': 0.0, 'featureimp_wilcox': 0.0}
{'featureimp_cor': 0.8679049272267945, 'featureimp_wcor': 0.8941975748626069}
{'featureimp_ks': 0.0, 'featureimp_wilcox': 0.0}


In [11]:
for i in model_name_list:
    print(calculate_position_predict(
        new_fadata,
        ref_model='ref',
        pred_model=i,
        metrics=["rf_mse","rf_rsq","rf_nmse","lm_mse","lm_rsq","lm_nmse"]
    ))

{'summary': {'rf_mse': 7.213010917162967e-05, 'rf_rsq': 0.9982256581527431, 'rf_nmse': 0.9990810714485362, 'lm_mse': 0.0, 'lm_rsq': 1.0, 'lm_nmse': 1.0}, 'rf_mses': {'Alpha': 1.8420746824911923e-05, 'Beta': 2.0956468787961036e-05, 'Delta': 0.00035783022419503234, 'Ductal': 0.00010619117126407761, 'Epsilon': 1.815906007833832e-05, 'Ngn3 high EP': 1.893423008471104e-05, 'Ngn3 low EP': 2.9246195611453932e-05, 'Pre-endocrine': 7.302776526551191e-06}, 'rf_rsqs': {'Alpha': 0.9998242692165579, 'Beta': 0.9998066664690053, 'Delta': 0.9882998366348373, 'Ductal': 0.9990724160998429, 'Epsilon': 0.9994652658208695, 'Ngn3 high EP': 0.9997470108808021, 'Ngn3 low EP': 0.9997158107732999, 'Pre-endocrine': 0.9998739893267301}, 'lm_rsqs': {'Alpha': 1.0, 'Beta': 1.0, 'Delta': 1.0, 'Ductal': 1.0, 'Epsilon': 1.0, 'Ngn3 high EP': 1.0, 'Ngn3 low EP': 1.0, 'Pre-endocrine': 1.0}}
{'summary': {'rf_mse': 0.06732725973111518, 'rf_rsq': 0.15447800416805574, 'rf_nmse': 0.1422591485127923, 'lm_mse': 0.067519534736153

拓扑部分

In [ ]:
model_metric_dict = {}
metric_id_list = metrics[metrics["category"] == "topology"]["metric_id"].tolist()
for model_name in model_name_list:
    model_metric = cfe.metric.calculate_metrics(
        fadata,
        metrics=metric_id_list,
        now_model=model_name,
        ref_model="ref"
    )
    model_metric_dict[model_name] = model_metric

df = pd.DataFrame(model_metric_dict).T
df.index=parsed_model_name_list
df

,isomorphic,edge_flip,him
ref,1.0,1.0,1.000000
comp1-python_function,0.0,0.0,0.272876
state_comp-python_function,0.0,0.4,0.212602
paga-python_function,0.0,0.0,0.537932
cluster_mst-python_function,0.0,0.0,0.272876
projection_mst-python_function,0.0,0.5,0.578087


F1 score等部分

In [13]:
#映射到里程碑上
F1_dict={}
for i,j in zip(model_name_list,parsed_model_name_list):
    F1_dict[j]=calculate_mapping(new_fadata,grouping='milestones',simplify=True,ref_model= "ref",pred_model= i)
    
F1_dict

{'ref': {'recovery': 1.0, 'relevance': 1.0, 'F1': 1.0},
 'comp1-python_function': {'recovery': 0.13143736204506956,
  'relevance': 0.18840579710144928,
  'F1': 0.1548481513945341},
 'state_comp-python_function': {'recovery': 0.12469512195121951,
  'relevance': 0.09310412381232247,
  'F1': 0.10660854248818853},
 'paga-python_function': {'recovery': 0.34324747725377286,
  'relevance': 0.4402029855447131,
  'F1': 0.38572589191683554},
 'cluster_mst-python_function': {'recovery': 1.0, 'relevance': 1.0, 'F1': 1.0},
 'projection_mst-python_function': {'recovery': 0.4688077857831513,
  'relevance': 0.47219320244981794,
  'F1': 0.47049440430034717}}

In [14]:
df = pd.DataFrame.from_dict(F1_dict, orient='index')
df

,recovery,relevance,F1
ref,1.000000,1.000000,1.000000
comp1-python_function,0.131437,0.188406,0.154848
state_comp-python_function,0.124695,0.093104,0.106609
paga-python_function,0.343247,0.440203,0.385726
cluster_mst-python_function,1.000000,1.000000,1.000000
projection_mst-python_function,0.468808,0.472193,0.470494


In [15]:
#映射到分支上
for i in model_name_list:
    print(calculate_mapping(
        new_fadata,
        grouping='branches',
        simplify=True,
        ref_model= "ref",
        pred_model= i
    ))

{'recovery': 1.0, 'relevance': 1.0, 'F1': 1.0}
{'recovery': 0.14285714285714285, 'relevance': 0.272, 'F1': 0.18732782369146006}
{'recovery': 0.19766283756461714, 'relevance': 0.33213589529379006, 'F1': 0.24783344862540782}
{'recovery': 0.4406325169790399, 'relevance': 0.5125292374085532, 'F1': 0.4738693025923582}
{'recovery': 0.5969574010796553, 'relevance': 0.6567236698662998, 'F1': 0.625415927984096}
{'recovery': 0.4270793748156389, 'relevance': 0.43089527349135065, 'F1': 0.42897883842333245}


查看correlation表现


In [16]:

model_metric_dict = {}
metric_id_list = metrics[metrics["category"] == "cell positions"]["metric_id"].tolist()
for model_name in model_name_list:
    model_metric = cfe.metric.calculate_metrics(
        fadata,
        metrics=metric_id_list,
        now_model=model_name,
        ref_model="ref"
    )
    model_metric_dict[model_name] = model_metric

df = pd.DataFrame(model_metric_dict).T
df.index=parsed_model_name_list
df

,correlation
ref,1.000000
comp1-python_function,0.197914
state_comp-python_function,0.000000
paga-python_function,0.000000
cluster_mst-python_function,0.000000
projection_mst-python_function,0.000000
